In [1]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


Trained model
      ↓
Inspect weights
      ↓
Weight statistics
      ↓
Weight heatmaps
      ↓
Determine low-importance weights
      ↓
Pruning
      ↓
Fine-tuning
      ↓
Evaluation
      ↓
Compare original vs pruned model

In [2]:
model = models.swin_t(weights=None)

num_classes = 7

in_features = model.head.in_features

model.head = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

print(model.head)

Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=768, out_features=7, bias=True)
)


In [3]:
model = models.swin_t(weights=None)

num_classes = 7

in_features = model.head.in_features

model.head = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, num_classes)
)

model = model.to(device)

print(model.head)


Sequential(
  (0): Dropout(p=0.3, inplace=False)
  (1): Linear(in_features=768, out_features=7, bias=True)
)


In [4]:
checkpoint_path = "../models/swin_tiny_best.pth"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("Checkpoint keys:")
    print(checkpoint.keys())

Checkpoint type: <class 'collections.OrderedDict'>
Checkpoint keys:
odict_keys(['features.0.0.weight', 'features.0.0.bias', 'features.0.2.weight', 'features.0.2.bias', 'features.1.0.norm1.weight', 'features.1.0.norm1.bias', 'features.1.0.attn.relative_position_bias_table', 'features.1.0.attn.relative_position_index', 'features.1.0.attn.qkv.weight', 'features.1.0.attn.qkv.bias', 'features.1.0.attn.proj.weight', 'features.1.0.attn.proj.bias', 'features.1.0.norm2.weight', 'features.1.0.norm2.bias', 'features.1.0.mlp.0.weight', 'features.1.0.mlp.0.bias', 'features.1.0.mlp.3.weight', 'features.1.0.mlp.3.bias', 'features.1.1.norm1.weight', 'features.1.1.norm1.bias', 'features.1.1.attn.relative_position_bias_table', 'features.1.1.attn.relative_position_index', 'features.1.1.attn.qkv.weight', 'features.1.1.attn.qkv.bias', 'features.1.1.attn.proj.weight', 'features.1.1.attn.proj.bias', 'features.1.1.norm2.weight', 'features.1.1.norm2.bias', 'features.1.1.mlp.0.weight', 'features.1.1.mlp.0.bias'

In [5]:
if isinstance(checkpoint, dict):

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]

    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

    else:
        state_dict = checkpoint

else:
    state_dict = checkpoint


model.load_state_dict(state_dict)

model.eval()

print("✅ Trained Swin-Tiny weights loaded successfully.")

✅ Trained Swin-Tiny weights loaded successfully.


In [6]:
print("=" * 80)
print("MODEL PARAMETERS")
print("=" * 80)

total_params = 0

for name, param in model.named_parameters():

    if param.requires_grad:

        num_params = param.numel()
        total_params += num_params

        print(
            f"{name:60s} "
            f"shape={str(tuple(param.shape)):25s} "
            f"params={num_params:,}"
        )

print("=" * 80)
print(f"Total trainable parameters: {total_params:,}")

MODEL PARAMETERS
features.0.0.weight                                          shape=(96, 3, 4, 4)             params=4,608
features.0.0.bias                                            shape=(96,)                     params=96
features.0.2.weight                                          shape=(96,)                     params=96
features.0.2.bias                                            shape=(96,)                     params=96
features.1.0.norm1.weight                                    shape=(96,)                     params=96
features.1.0.norm1.bias                                      shape=(96,)                     params=96
features.1.0.attn.relative_position_bias_table               shape=(169, 3)                  params=507
features.1.0.attn.qkv.weight                                 shape=(288, 96)                 params=27,648
features.1.0.attn.qkv.bias                                   shape=(288,)                    params=288
features.1.0.attn.proj.weight                  

In [7]:
print("=" * 80)
print("MODEL PARAMETERS")
print("=" * 80)

total_params = 0

for name, param in model.named_parameters():

    if param.requires_grad:

        num_params = param.numel()
        total_params += num_params

        print(
            f"{name:60s} "
            f"shape={str(tuple(param.shape)):25s} "
            f"params={num_params:,}"
        )

print("=" * 80)
print(f"Total trainable parameters: {total_params:,}")

MODEL PARAMETERS
features.0.0.weight                                          shape=(96, 3, 4, 4)             params=4,608
features.0.0.bias                                            shape=(96,)                     params=96
features.0.2.weight                                          shape=(96,)                     params=96
features.0.2.bias                                            shape=(96,)                     params=96
features.1.0.norm1.weight                                    shape=(96,)                     params=96
features.1.0.norm1.bias                                      shape=(96,)                     params=96
features.1.0.attn.relative_position_bias_table               shape=(169, 3)                  params=507
features.1.0.attn.qkv.weight                                 shape=(288, 96)                 params=27,648
features.1.0.attn.qkv.bias                                   shape=(288,)                    params=288
features.1.0.attn.proj.weight                  

In [8]:
print("=" * 100)
print("WEIGHT STATISTICS")
print("=" * 100)

for name, param in model.named_parameters():

    if "weight" in name and param.requires_grad:

        w = param.detach().cpu()

        print(f"\nLayer: {name}")
        print(f"Shape       : {tuple(w.shape)}")
        print(f"Min         : {w.min().item():.6f}")
        print(f"Max         : {w.max().item():.6f}")
        print(f"Mean        : {w.mean().item():.6f}")
        print(f"Std         : {w.std().item():.6f}")
        print(f"Mean |W|    : {w.abs().mean().item():.6f}")
        print(f"Zero values : {(w == 0).sum().item():,}")
        print(f"Total values: {w.numel():,}")

WEIGHT STATISTICS

Layer: features.0.0.weight
Shape       : (96, 3, 4, 4)
Min         : -0.258799
Max         : 0.242027
Mean        : 0.001373
Std         : 0.056364
Mean |W|    : 0.043125
Zero values : 0
Total values: 4,608

Layer: features.0.2.weight
Shape       : (96,)
Min         : -0.013618
Max         : 4.251256
Mean        : 1.443248
Std         : 1.341469
Mean |W|    : 1.444817
Zero values : 0
Total values: 96

Layer: features.1.0.norm1.weight
Shape       : (96,)
Min         : -0.002233
Max         : 1.578722
Mean        : 0.718102
Std         : 0.415784
Mean |W|    : 0.718242
Zero values : 0
Total values: 96

Layer: features.1.0.attn.qkv.weight
Shape       : (288, 96)
Min         : -0.533043
Max         : 0.531348
Mean        : 0.000940
Std         : 0.072706
Mean |W|    : 0.046236
Zero values : 0
Total values: 27,648

Layer: features.1.0.attn.proj.weight
Shape       : (96, 96)
Min         : -0.298414
Max         : 0.288377
Mean        : -0.002083
Std         : 0.050904
Mean 

In [9]:
# ============================================================
# PHASE 23A — WEIGHT MAGNITUDE ANALYSIS
# Memory-efficient percentile calculation
# ============================================================

import torch

# Collect a sample of absolute weight values from each layer
weight_samples = []

MAX_SAMPLES_PER_LAYER = 100_000

for name, param in model.named_parameters():

    if "weight" in name and param.requires_grad:

        w = param.detach().cpu().abs().flatten()

        # If layer is large, randomly sample from it
        if w.numel() > MAX_SAMPLES_PER_LAYER:
            indices = torch.randperm(w.numel())[:MAX_SAMPLES_PER_LAYER]
            w = w[indices]

        weight_samples.append(w)

# Combine only the samples
sampled_weights = torch.cat(weight_samples)

print("Number of sampled weights:", sampled_weights.numel())

print("\nEstimated weight magnitude percentiles:")
print("-" * 60)

for percentile in [1, 5, 10, 20, 30, 40, 50]:

    threshold = torch.quantile(
        sampled_weights,
        percentile / 100
    )

    print(
        f"{percentile:2d}% smallest weights "
        f"have magnitude <= {threshold.item():.8f}"
    )

Number of sampled weights: 4391008

Estimated weight magnitude percentiles:
------------------------------------------------------------
 1% smallest weights have magnitude <= 0.00054209
 5% smallest weights have magnitude <= 0.00273319
10% smallest weights have magnitude <= 0.00557540
20% smallest weights have magnitude <= 0.01154655
30% smallest weights have magnitude <= 0.01783089
40% smallest weights have magnitude <= 0.02453476
50% smallest weights have magnitude <= 0.03187289


In [10]:
layer_stats = []

for name, param in model.named_parameters():

    if "weight" in name and param.requires_grad:

        w = param.detach().cpu()

        abs_w = w.abs()

        layer_stats.append({
            "layer": name,
            "parameters": w.numel(),
            "mean_abs_weight": abs_w.mean().item(),
            "median_abs_weight": abs_w.median().item(),
            "min_abs_weight": abs_w.min().item(),
            "max_abs_weight": abs_w.max().item(),
            "zero_count": (w == 0).sum().item()
        })

import pandas as pd

layer_df = pd.DataFrame(layer_stats)

layer_df

,layer,parameters,mean_abs_weight,median_abs_weight,min_abs_weight,max_abs_weight,zero_count
0,features.0.0.weight,4608,0.043125,0.034531,6.767704e-06,0.258799,0
1,features.0.2.weight,96,1.444817,0.966776,1.855024e-03,4.251256,0
2,features.1.0.norm1.weight,96,0.718242,0.774582,4.753127e-05,1.578722,0
3,features.1.0.attn.qkv.weight,27648,0.046236,0.026562,1.894096e-07,0.533043,0
4,features.1.0.attn.proj.weight,9216,0.037092,0.026738,3.490932e-06,0.298414,0
...,...,...,...,...,...,...,...
77,features.7.1.norm2.weight,768,2.354996,2.273113,1.813582e-03,5.360744,0
78,features.7.1.mlp.0.weight,2359296,0.038768,0.032374,1.448017e-08,0.584206,0
79,features.7.1.mlp.3.weight,2359296,0.041720,0.034690,4.256801e-09,1.638299,0
80,norm.weight,768,2.008732,2.004053,9.858832e-04,4.594053,0
